In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor

In [2]:
df=pd.read_csv('../Data/train-test.csv')
validation=pd.read_csv('../Data/validation.csv')

# Note:
#### Negative weight values appear to follow a reversed encoding pattern.
#### For negative values, the cleaned weight is calculated as 47,500 + weight,
#### while positive values remain unchanged.

In [3]:
df["weight_clean"] = df["weight"]

negative_mask = df["weight"] < 0

df.loc[negative_mask, "weight_clean"] = (
    47500 + df.loc[negative_mask, "weight"]
)

In [4]:
validation["weight_clean"] = validation["weight"]

negative_mask = validation["weight"] < 0

validation.loc[negative_mask, "weight_clean"] = (
    47500 + validation.loc[negative_mask, "weight"]
)

In [5]:
# Fill missing weight values with the median
median_weight = df["weight_clean"].median()

df["weight_clean"] = df["weight_clean"].fillna(median_weight)
validation["weight_clean"] = validation["weight_clean"].fillna(median_weight)


In [6]:
# Calculate the daily average market index
daily_market_mean = df.groupby("date")["market_index"].transform("mean")
daily_market_meanva=validation.groupby("date")["market_index"].transform("mean")

# Fill missing values with the average of the same day
df["market_index"] = df["market_index"].fillna(daily_market_mean)
validation["market_index"] = validation["market_index"].fillna(daily_market_meanva)

In [7]:
df["date"] = pd.to_datetime(df["date"])
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['week'] = df['date'].dt.isocalendar().week.astype(int)


In [8]:
validation["date"] = pd.to_datetime(validation["date"])
validation['month'] = validation['date'].dt.month
validation['day'] = validation['date'].dt.day
validation['week'] = validation['date'].dt.isocalendar().week.astype(int)

In [9]:
df = df.drop(columns=["load_id","weight","date"])
validation_1 = validation.drop(columns=["load_id","weight","date"])

In [10]:
numerical_features = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "weight_clean",
    "market_index",
    "quote_signal",
    "month",
    "day"
]

categorical_features = [
    "pickup",
    "delivery",
    "equipment"
]

#Modeling

In [11]:
X_train = df.drop("posted_rate", axis=1)
y_train = df["posted_rate"]

X_val = validation_1


In [12]:
cat_features = ["pickup", "delivery", "equipment"]

In [13]:
final_model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.01,
    depth=4,
    l2_leaf_reg=5,
    random_strength=1,
    bagging_temperature=0,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    verbose=100
)

final_model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    early_stopping_rounds=150
)

0:	learn: 1119.8610988	total: 187ms	remaining: 9m 19s
100:	learn: 495.8924259	total: 5.11s	remaining: 2m 26s
200:	learn: 274.6292188	total: 9.05s	remaining: 2m 5s
300:	learn: 180.7860449	total: 12.5s	remaining: 1m 52s
400:	learn: 140.0308699	total: 16.1s	remaining: 1m 44s
500:	learn: 123.2200176	total: 19.9s	remaining: 1m 39s
600:	learn: 113.4384438	total: 29.3s	remaining: 1m 56s
700:	learn: 107.4869074	total: 33.3s	remaining: 1m 49s
800:	learn: 103.5043970	total: 37.6s	remaining: 1m 43s
900:	learn: 100.8136084	total: 42.7s	remaining: 1m 39s
1000:	learn: 98.6923847	total: 46.7s	remaining: 1m 33s
1100:	learn: 97.0921679	total: 51.4s	remaining: 1m 28s
1200:	learn: 95.8945541	total: 1m 2s	remaining: 1m 33s
1300:	learn: 94.6064032	total: 1m 8s	remaining: 1m 30s
1400:	learn: 93.4021333	total: 1m 12s	remaining: 1m 23s
1500:	learn: 92.4390250	total: 1m 17s	remaining: 1m 16s
1600:	learn: 91.6224245	total: 1m 21s	remaining: 1m 10s
1700:	learn: 90.8770302	total: 1m 25s	remaining: 1m 5s
1800:	lea

CatBoostRegressor(bagging_temperature=0, depth=4, eval_metric='MAE', iterations=3000, l2_leaf_reg=5, learning_rate=0.01, loss_function='MAE', random_seed=42, random_strength=1, verbose=100)

In [14]:
predictions = final_model.predict(X_val)

result = pd.DataFrame({
    "load_id": validation["load_id"],
    "posted_rate": predictions
})

result.to_csv("../Data/final_predictions_1.csv", index=False)

In [15]:
dec = pd.read_csv("../Data/december-chart-inputs.csv")

# Route coordinates from the training data
route = df[
    (df["pickup"] == "Lexington") &
    (df["delivery"] == "Fort Wayne")
].iloc[0]

# Build the features required by the model
dec["pickup_lat"] = route["pickup_lat"]
dec["pickup_lon"] = route["pickup_lon"]
dec["delivery_lat"] = route["delivery_lat"]
dec["delivery_lon"] = route["delivery_lon"]

dec["weight_clean"] = dec["weight"]

dec["month"] = pd.to_datetime(dec["date"]).dt.month
dec["day"] = pd.to_datetime(dec["date"]).dt.day
dec["week"] = pd.to_datetime(dec["date"]).dt.isocalendar().week.astype(int)

# Use training medians for features missing from the December template
dec["market_index"] = df["market_index"].median()
dec["quote_signal"] = df["quote_signal"].median()

# Exactly the same feature order used by the model
features = [
    "pickup",
    "delivery",
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "equipment",
    "market_index",
    "quote_signal",
    "weight_clean",
    "month",
    "day",
    "week"
]

X_dec = dec[features]

# Predict
dec["predicted_rate"] = final_model.predict(X_dec)

# Save the required December file
output = dec[
    ["pickup", "delivery", "distance", "equipment",
     "weight", "date", "predicted_rate"]
]

output.to_csv("../Data/december_predictions.csv", index=False)

print(output)
print("\nSaved: december_predictions.csv")

       pickup    delivery  distance equipment  weight        date  \
0   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-01   
1   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-02   
2   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-03   
3   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-04   
4   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-05   
5   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-06   
6   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-07   
7   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-08   
8   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-09   
9   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-10   
10  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-11   
11  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-12   
12  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-13   
13  Lexington  Fort Wayne       36